In [3]:
import simnibs
from simnibs import mesh_io
import numpy as np
import nibabel as nib
import pandas as pd
import os
#from scipy.spatial.distance import cdist
from simnibs.simulation import electrode_placement
#from simnibs.cli import msh2nii
import subprocess

In [2]:
def create_3x3_array(center, normal):
    """Create a 3x3 grid of electrodes on a plane"""
    if abs(normal[0]) > 0.5:
        temp = np.array([0, 1, 0])
    else:
        temp = np.array([1, 0, 0])
    
    tangent1 = np.cross(normal, temp)
    tangent1 /= np.linalg.norm(tangent1)
    tangent2 = np.cross(normal, tangent1)
    
    positions = []
    for i in [-1, 0, 1]:
        for j in [-1, 0, 1]:
            offset = (i * 25 * tangent1) + (j * 27 * tangent2)
            positions.append(center + offset)
    
    return np.array(positions)

In [3]:
def get_random_scalp_locations(mesh, min_distance=100, max_attempts=100):
    """Get two random scalp locations that aren't too close"""
    scalp_surface = mesh.crop_mesh(1005)
    nodes = scalp_surface.nodes.node_coord
    normals = scalp_surface.nodes_normals()

   # Get min/max for Z (height) and Y (depth)
    z_min, z_max = np.min(nodes[:, 2]), np.max(nodes[:, 2])  # Z: inferior-superior
    y_min, y_max = np.min(nodes[:, 1]), np.max(nodes[:, 1])  # Y: posterior-anterior
    
    # Define thresholds as percentages (e.g., 30% from bottom, 20% from front)
    z_threshold = z_min + 0.55 * (z_max - z_min)  # Keep nodes above 45% of head height
    y_threshold = y_min + 0.20 * (y_max - y_min)  # Keep nodes behind 20% of head depth
    z2_threshold = z_min + 35
    
    # Combined condition: above eyebrows OR behind ears and with normal not directed into -z
    mask = ((nodes[:, 2] > z_threshold) | ((nodes[:, 1] < y_threshold) & (nodes[:, 2] > z2_threshold)))
    # Filter nodes and normals
    filtered_nodes = nodes[mask]
    filtered_normals = normals[mask]

    
    # Pick first random location
    idx1 = np.random.choice(len(filtered_nodes))
    center1 = filtered_nodes[idx1]
    normal1 = -filtered_normals[idx1]  # Flip normal inward
    normal1 /= np.linalg.norm(normal1)
    
    # Pick second location that's sufficiently far away
    attempts = 0
    while attempts < max_attempts:
        idx2 = np.random.choice(len(filtered_nodes))
        center2 = filtered_nodes[idx2]
        
        # Check distance between array centers
        if np.linalg.norm(center1 - center2) >= min_distance:
            normal2 = -filtered_normals[idx2]
            normal2 /= np.linalg.norm(normal2)
            return [(center1, normal1), (center2, normal2)]
        
        attempts += 1
    
    # If we can't find a good pair, just return any two locations
    idx2 = (idx1 + len(nodes)//2) % len(nodes)  # At least pick something opposite-ish
    center2 = nodes[idx2]
    normal2 = -normals[idx2]
    normal2 /= np.linalg.norm(normal2)
    return [(center1, normal1), (center2, normal2)]

In [4]:
def add_electrode_arrays(mesh, array_locations):
    """Add electrode arrays to the mesh"""
    #all_electrodes_poses = []
    for array_idx, (center, normal) in enumerate(array_locations):
        positions = create_3x3_array(center, normal)
        
        for i, pos in enumerate(positions):
            el = simnibs.simulation.sim_struct.ELECTRODE()
            el.centre = pos
            el.shape = 'ellipse'
            el.dimensions = [20, 20]  # 20mm diameter
            el.thickness = [2 , 2]  # 2mm gel 2mm elecrode
            #all_electrodes_poses.append(pos)
            mesh_with_elec , tag = electrode_placement.put_electrode_on_mesh(
                el, mesh, 
                array_idx + 2000) #tag
    
    return mesh_with_elec 

In [294]:
df = pd.read_csv("E:/Datasets/Stage 2/add electrode folder/Subjects 7.csv")
subject_id_column = 'Subject ID'
for subject_id in df[subject_id_column]:
    input_mesh_path = f"E:/Datasets/Stage 2/Meshes/m2m_{subject_id}/{subject_id}.msh"
    output_base_dir = "E:/Datasets/Stage 2/mesh with electrode"
    output_dir = os.path.join(output_base_dir, subject_id)
    os.makedirs(output_dir, exist_ok=True)
    
    
    ref_nii_path = f"E:/Datasets/Stage 2/Meshes/m2m_{subject_id}/final_tissues.nii.gz"
    ref_img = nib.load(ref_nii_path)
    
    result_label_bas_dir = "E:/Datasets/Stage 2/labeled nifti with electrodes"
    result_label = os.path.join(result_label_bas_dir, subject_id)
    os.makedirs(result_label, exist_ok=True)
    for i in range(1, 6):
        mesh = mesh_io.read_msh(input_mesh_path)
        array_locations = get_random_scalp_locations(mesh)
        mesh_with_elec = add_electrode_arrays(mesh, array_locations)
    
        # Save final model
        final_path = os.path.join(output_dir, f'mesh_{i}.msh')
        mesh_io.write_msh(mesh_with_elec, final_path)
    
        output_label = os.path.join(result_label, f'labels_{i}.nii.gz')
    
        command = f'msh2nii "{final_path}" "{ref_nii_path}" "{output_label}" --create_label'
        subprocess.run(command)
        
    print(f"Created 5 electrode array {output_dir}")

Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-14-3477
Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-19-0963
Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-19-1789
Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-19-2624


E:\Anaconda\envs\simnibs_env\lib\site-packages\simnibs\simulation\electrode_placement.py:269: UserWarning: Found an edge with more than 2 adjacent triangles!
  warnings.warn(


Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-19-2631
Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-19-5951
Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-19-5954
Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-19-5958
Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-19-5960
Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-76-4932
Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-76-4934
Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-76-4935
Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-76-6191
Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-76-6193
Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-76-6280
Created 5 electrode array E:/Datasets/Stage 2/mesh with electrode\TCGA-76-6282
Created 5 electrode array E:/Datasets/Stage 2/mesh w

In [152]:
mesh_path = "E:/Datasets/Stage 2/mesh with electrode/TCGA-19-5951/mesh_5.msh"
result_mesh = mesh_io.read_msh(mesh_path)
#Electrode = result_mesh.crop_mesh(3400)
v = result_mesh.view()
v.show()

In [151]:
#this to remake specific mesh who failed to add arrays right
subject_id = "TCGA-19-5951"
i = 5
input_mesh_path = f"E:/Datasets/Stage 2/Meshes/m2m_{subject_id}/{subject_id}.msh"
output_base_dir = "E:/Datasets/Stage 2/mesh with electrode"
output_dir = os.path.join(output_base_dir, subject_id)
os.makedirs(output_dir, exist_ok=True)


ref_nii_path = f"E:/Datasets/Stage 2/Meshes/m2m_{subject_id}/final_tissues.nii.gz"
ref_img = nib.load(ref_nii_path)

result_label_bas_dir = "E:/Datasets/Stage 2/labeled nifti with electrodes"
result_label = os.path.join(result_label_bas_dir, subject_id)
os.makedirs(result_label, exist_ok=True)

mesh = mesh_io.read_msh(input_mesh_path)
array_locations = get_random_scalp_locations(mesh)
mesh_with_elec = add_electrode_arrays(mesh, array_locations)

# Save final model
final_path = os.path.join(output_dir, f'mesh_{i}.msh')
mesh_io.write_msh(mesh_with_elec, final_path)

output_label = os.path.join(result_label, f'labels_{i}.nii.gz')

command = f'msh2nii "{final_path}" "{ref_nii_path}" "{output_label}" --create_label'
subprocess.run(command)
print("done")

done


In [5]:
help(simnibs.simulation.fem.FEMSystem)

Help on class FEMSystem in module simnibs.simulation.fem:

class FEMSystem(builtins.object)
 |  FEMSystem(mesh, cond, dirichlet=None, units='mm', store_G=False, solver_options=None)
 |  
 |  Class defining the equation system to be used in FEM calculations
 |  
 |  Parameters
 |  ------------
 |  mesh: mesh_io.Msh
 |     Mesh object
 |  cond: mesh_io.ElementData
 |      conductivity information
 |  dirichlet: list of DirichletBC (optional)
 |      Dirichlet boundary conditions
 |  units: {'mm' or 'm'} (optional)
 |      Units of the mesh nodes. Default: mm
 |  store_G: bool (optional)
 |      Wether to store the gradient matrix. Default: False
 |  solver_options: str
 |      Options to be used by the solver. Default: DEFAULT_SOLVER_OPTIONS
 |  Attributes
 |  ------------
 |  mesh: mesh_io.Msh
 |     Mesh object
 |  cond: mesh_io.ElementData
 |      conductivity information
 |  dirichlet: DirichletBC (optional)
 |      Dirichlet boundary condition
 |  units: {'mm' or 'm'}
 |      Units 

In [33]:
from simnibs import opt_struct

In [3]:
help(mesh_io.ElementData)

Help on class ElementData in module simnibs.mesh_tools.mesh_io:

class ElementData(Data)
 |  ElementData(value, name='', mesh=None)
 |  
 |  Data (scalar, vector or tensor) defined in mesh elements.
 |  
 |  Parameters
 |  -----------------------
 |  value: ndarray
 |      Value of field in elements. Should have the shape
 |          - (n_elm,) or (n_elm, 1) for scalar fields
 |          - (n_elm, 3) for vector fields
 |          - (n_elm, 9) for tensors
 |  
 |  field_name: str (optional)
 |      name of field. Default: ''
 |  
 |  mesh: simnibs.msh.Msh (optional)
 |      Mesh structure where the field is defined. Required for many methods
 |  
 |  
 |  Attributes
 |  --------------------------
 |  value: ndarray
 |      Value of field in elements
 |  field_name: str
 |      name of field
 |  elm_number: ndarray
 |      index of elements
 |  nr: property
 |      number of data points
 |  nr_comp: property
 |      number of dimensions per data point (1 for scalars, 3 for vectors)
 |  


In [5]:
help(simnibs.simulation.sim_struct.cond)

Help on module simnibs.simulation.cond in simnibs.simulation:

NAME
    simnibs.simulation.cond

DESCRIPTION
    Manipulation of conductivity information for SimNIBS
    This program is part of the SimNIBS package.
    Please check on www.simnibs.org how to cite our work in publications.
    
    Copyright (C) 2018 Guilherme Saturnino, Andre Antunes, Axel Thielscher
    
    This program is free software: you can redistribute it and/or modify
    it under the terms of the GNU General Public License as published by
    the Free Software Foundation, either version 3 of the License, or any later version.
    
    This program is distributed in the hope that it will be useful,
    but WITHOUT ANY WARRANTY; without even the implied warranty of
    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
    GNU General Public License for more details.
    
    You should have received a copy of the GNU General Public License
    along with this program.  If not, see <http://www.gnu.org